In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, roc_auc_score, balanced_accuracy_score,
    precision_score, recall_score
)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
import warnings
import joblib

warnings.filterwarnings('ignore')

# -------------------------------
# Load dataset
# -------------------------------
disease = pd.read_csv("heart_2020_cleaned.csv", low_memory=False)
print("Dataset loaded - Shape:", disease.shape)

# -------------------------------
# 1. CONVERT BINARY COLUMNS FIRST
# -------------------------------
binary_columns = ['Smoking', 'AlcoholDrinking', 'Stroke', 'DiffWalking', 
                  'Asthma', 'KidneyDisease', 'SkinCancer']

for col in binary_columns:
    disease[col] = disease[col].map({'Yes': 1, 'No': 0})

# -------------------------------
# 2. CREATE AGE NUMERIC
# -------------------------------
age_map = {
    '18-24': 1, '25-29': 2, '30-34': 3, '35-39': 4, '40-44': 5,
    '45-49': 6, '50-54': 7, '55-59': 8, '60-64': 9, '65-69': 10,
    '70-74': 11, '75-79': 12, '80 or older': 13
}
disease['Age_Numeric'] = disease['AgeCategory'].map(age_map)

# -------------------------------
# 3. CREATE DIABETIC SCORE (ENHANCED)
# -------------------------------
def map_diabetic(val):
    if val == 'Yes':
        return 15
    elif val == 'No':
        return 0
    elif val == 'No, borderline diabetes':
        return 7
    elif val == 'Yes (during pregnancy)':
        return 5
    else:
        return 0

disease['Diabetic_Score'] = disease['Diabetic'].apply(map_diabetic)

# -------------------------------
# 4. ENHANCED FEATURE ENGINEERING
# -------------------------------
# Basic interactions
disease['BMI_Age'] = disease['BMI'] * disease['Age_Numeric']
disease['Physical_to_Mental'] = disease['PhysicalHealth'] / (disease['MentalHealth'] + 1)

# Age-Specific Risk Factors
disease['Elderly_Risk'] = (disease['Age_Numeric'] >= 11).astype(int)
disease['Middle_Age_Risk'] = ((disease['Age_Numeric'] >= 8) & (disease['Age_Numeric'] <= 10)).astype(int)

# Comorbidity Score
disease['Comorbidity_Score'] = (
    disease['Stroke'] + 
    disease['Asthma'] + 
    disease['KidneyDisease'] + 
    disease['SkinCancer'] +
    (disease['Diabetic_Score'] > 0).astype(int)
)

# Enhanced Health Score
disease['Enhanced_Health_Score'] = (
    disease['PhysicalHealth'] * 2 + 
    disease['MentalHealth'] * 1.5 + 
    disease['DiffWalking'] * 25 + 
    disease['Smoking'] * 15 +
    disease['AlcoholDrinking'] * 8 + 
    disease['Stroke'] * 40 +
    disease['Diabetic_Score'] * 1.5 +
    disease['KidneyDisease'] * 30 + 
    disease['SkinCancer'] * 25 +
    disease['Asthma'] * 12 +
    disease['Comorbidity_Score'] * 20
)

# BMI Risk Categories
def bmi_risk_category(bmi):
    if bmi < 18.5: return 0
    elif bmi < 25: return 1
    elif bmi < 30: return 2
    elif bmi < 35: return 3
    elif bmi < 40: return 4
    else: return 5

disease['BMI_Risk'] = disease['BMI'].apply(bmi_risk_category)

# Sleep Quality Score
disease['Sleep_Quality'] = ((disease['SleepTime'] >= 7) & (disease['SleepTime'] <= 9)).astype(int)

# Physical and Mental Health Severity
disease['Physical_Severe'] = (disease['PhysicalHealth'] >= 15).astype(int)
disease['Mental_Severe'] = (disease['MentalHealth'] >= 15).astype(int)

# Age-BMI Interaction Squared
disease['Age_BMI_Squared'] = disease['Age_Numeric'] * (disease['BMI'] ** 2) / 100

# Total Health Days Impact
disease['Total_Poor_Health_Days'] = disease['PhysicalHealth'] + disease['MentalHealth']
disease['High_Health_Impact'] = (disease['Total_Poor_Health_Days'] >= 30).astype(int)

# Risk Score
disease['Risk_Score'] = (
    disease['Age_Numeric'] * 15 +
    disease['BMI'] * 8 +
    (disease['Smoking'] * 25) +
    (disease['Stroke'] * 35) +
    disease['Diabetic_Score'] * 10 +
    (disease['DiffWalking'] * 20) +
    disease['Comorbidity_Score'] * 15
)

disease['Poor_Sleep'] = (disease['SleepTime'] < 6).astype(int)

# -------------------------------
# 5. DROP ORIGINAL COLUMNS
# -------------------------------
disease = disease.drop(['AgeCategory', 'Diabetic'], axis=1)

# -------------------------------
# 6. PREPARE FEATURES AND TARGET
# -------------------------------
categorical_cols = ['Sex', 'Race', 'GenHealth']

numerical_cols = [
    'BMI', 'BMI_Age', 'PhysicalHealth', 'MentalHealth', 
    'Physical_to_Mental', 'Poor_Sleep', 
    'BMI_Risk', 'Age_Numeric', 'SleepTime', 'Diabetic_Score',
    'Elderly_Risk', 'Middle_Age_Risk', 'Comorbidity_Score',
    'Enhanced_Health_Score', 'Sleep_Quality', 'Physical_Severe',
    'Mental_Severe', 'Age_BMI_Squared', 'Total_Poor_Health_Days',
    'High_Health_Impact', 'Risk_Score'
]

binary_cols = [
    'Smoking', 'AlcoholDrinking', 'Stroke', 'DiffWalking', 
    'Asthma', 'KidneyDisease', 'SkinCancer'
]

# Verify all columns exist
categorical_cols = [c for c in categorical_cols if c in disease.columns]
numerical_cols = [c for c in numerical_cols if c in disease.columns]
binary_cols = [c for c in binary_cols if c in disease.columns]

print(f"\nFeatures:")
print(f"  Numerical: {len(numerical_cols)}")
print(f"  Categorical: {len(categorical_cols)}")
print(f"  Binary: {len(binary_cols)}")
print(f"  Total Features: {len(numerical_cols) + len(categorical_cols) + len(binary_cols)}")

# -------------------------------
# 7. PREPROCESSING PIPELINE
# -------------------------------
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols),
    ('bin', 'passthrough', binary_cols)
])

# -------------------------------
# 8. SPLIT FEATURES AND TARGET
# -------------------------------
X = disease.drop('HeartDisease', axis=1)
y = (disease['HeartDisease'] == 'Yes').astype(int)

print(f"\nTarget distribution:")
print(f"Class 0 (No): {(y==0).sum():,}")
print(f"Class 1 (Yes): {(y==1).sum():,}")
print(f"Imbalance ratio: {(y==1).sum()/(y==0).sum():.3f}")

# -------------------------------
# 9. APPLY PREPROCESSING
# -------------------------------
X_processed = preprocessor.fit_transform(X)
print(f"\nProcessed features shape: {X_processed.shape}")

# -------------------------------
# 10. ADVANCED IMBALANCE HANDLING - SMOTETomek
# -------------------------------
smote_tomek = SMOTETomek(random_state=42, sampling_strategy=0.6)
X_resampled, y_resampled = smote_tomek.fit_resample(X_processed, y)

print(f"\nAfter SMOTETomek:")
print(f"Class 0: {(y_resampled==0).sum():,}")
print(f"Class 1: {(y_resampled==1).sum():,}")

# -------------------------------
# 11. TRAIN-TEST SPLIT
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42
)

print(f"\nTrain set size: {X_train.shape[0]:,}")
print(f"Test set size: {X_test.shape[0]:,}")

# -------------------------------
# 12. ENHANCED MODELS
# -------------------------------
lgbm = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=45,
    max_depth=9,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=0.05,
    min_child_samples=15,
    min_child_weight=0.001,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    boosting_type='dart'
)

xgb = XGBClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=0.05,
    scale_pos_weight=3,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    n_jobs=-1
)

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

# -------------------------------
# 13. TRAIN INDIVIDUAL MODELS FIRST (FOR METRICS)
# -------------------------------
print("\n" + "="*70)
print("TRAINING INDIVIDUAL MODELS...")
print("="*70)

print("\nTraining LightGBM...")
lgbm.fit(X_train, y_train)

print("Training XGBoost...")
xgb.fit(X_train, y_train)

print("Training Random Forest...")
rf.fit(X_train, y_train)

# -------------------------------
# 14. STACKING ENSEMBLE
# -------------------------------
print("\nBuilding stacking ensemble...")

stacking_model = StackingClassifier(
    estimators=[
        ('lgbm', lgbm),
        ('xgb', xgb),
        ('rf', rf)
    ],
    final_estimator=LogisticRegression(C=1, max_iter=1000),
    cv=5,
    stack_method='predict_proba'
)

print("Training stacking ensemble (this may take a few minutes)...")
stacking_model.fit(X_train, y_train)

# -------------------------------
# 15. PREDICTIONS FROM STACKING ENSEMBLE
# -------------------------------
y_prob_stack = stacking_model.predict_proba(X_test)[:, 1]
y_pred_stack = stacking_model.predict(X_test)

# -------------------------------
# 16. TABLE 1: STACKING ENSEMBLE RESULTS (DEFAULT THRESHOLD 0.5)
# -------------------------------
print("\n" + "="*70)
print("TABLE 1: STACKING ENSEMBLE RESULTS (DEFAULT THRESHOLD = 0.5)")
print("="*70)

acc_stack = accuracy_score(y_test, y_pred_stack)
bal_acc_stack = balanced_accuracy_score(y_test, y_pred_stack)
f1_stack = f1_score(y_test, y_pred_stack)
auc_stack = roc_auc_score(y_test, y_prob_stack)
prec_stack = precision_score(y_test, y_pred_stack)
rec_stack = recall_score(y_test, y_pred_stack)
cm_stack = confusion_matrix(y_test, y_pred_stack)
spec_stack = cm_stack[0,0] / (cm_stack[0,0] + cm_stack[0,1])

print(f"  Accuracy:           {acc_stack:.4f} ({acc_stack*100:.2f}%)")
print(f"  Balanced Accuracy:  {bal_acc_stack:.4f} ({bal_acc_stack*100:.2f}%)")
print(f"  F1-Score:           {f1_stack:.4f} ({f1_stack*100:.2f}%)")
print(f"  ROC-AUC:            {auc_stack:.4f} ({auc_stack*100:.2f}%)")
print(f"  Precision:          {prec_stack:.4f} ({prec_stack*100:.2f}%)")
print(f"  Sensitivity/Recall: {rec_stack:.4f} ({rec_stack*100:.2f}%)")
print(f"  Specificity:        {spec_stack:.4f} ({spec_stack*100:.2f}%)")

# -------------------------------
# 17. THRESHOLD TUNING FOR ACCURACY
# -------------------------------
print("\n" + "="*70)
print("THRESHOLD TUNING FOR MAX ACCURACY")
print("="*70)

thresholds = np.arange(0.45, 0.75, 0.02)
best_thresh_acc = 0.5
best_acc = 0

for t in thresholds:
    y_pred_temp = (y_prob_stack >= t).astype(int)
    acc = accuracy_score(y_test, y_pred_temp)
    f1 = f1_score(y_test, y_pred_temp)
    print(f"Threshold: {t:.2f} | Accuracy: {acc:.4f} | F1: {f1:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_thresh_acc = t

print(f"\nBest threshold for ACCURACY: {best_thresh_acc:.2f} (Accuracy: {best_acc:.4f})")

# -------------------------------
# 18. TABLE 2: OPTIMIZED STACKING ENSEMBLE (BEST THRESHOLD)
# -------------------------------
y_pred_opt = (y_prob_stack >= best_thresh_acc).astype(int)

print("\n" + "="*70)
print(f"TABLE 2: OPTIMIZED STACKING ENSEMBLE (THRESHOLD = {best_thresh_acc:.2f})")
print("="*70)

acc_opt = accuracy_score(y_test, y_pred_opt)
bal_acc_opt = balanced_accuracy_score(y_test, y_pred_opt)
f1_opt = f1_score(y_test, y_pred_opt)
prec_opt = precision_score(y_test, y_pred_opt)
rec_opt = recall_score(y_test, y_pred_opt)
cm_opt = confusion_matrix(y_test, y_pred_opt)
spec_opt = cm_opt[0,0] / (cm_opt[0,0] + cm_opt[0,1])

print(f"  Accuracy:           {acc_opt:.4f} ({acc_opt*100:.2f}%)")
print(f"  Balanced Accuracy:  {bal_acc_opt:.4f} ({bal_acc_opt*100:.2f}%)")
print(f"  F1-Score:           {f1_opt:.4f} ({f1_opt*100:.2f}%)")
print(f"  ROC-AUC:            {auc_stack:.4f} ({auc_stack*100:.2f}%)")
print(f"  Precision:          {prec_opt:.4f} ({prec_opt*100:.2f}%)")
print(f"  Sensitivity/Recall: {rec_opt:.4f} ({rec_opt*100:.2f}%)")
print(f"  Specificity:        {spec_opt:.4f} ({spec_opt*100:.2f}%)")
print(f"  Optimal Threshold:  {best_thresh_acc:.2f}")

# -------------------------------
# 19. TABLE 3: INDIVIDUAL MODEL RESULTS (FULL METRICS)
# -------------------------------
print("\n" + "="*70)
print("TABLE 3: INDIVIDUAL MODEL PERFORMANCE (FULL METRICS)")
print("="*70)

def get_model_metrics(model, name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, y_pred)
    
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    spec = cm[0,0] / (cm[0,0] + cm[0,1])
    
    print(f"\n{name}:")
    print(f"  Accuracy:           {acc:.4f} ({acc*100:.2f}%)")
    print(f"  Balanced Accuracy:  {bal_acc:.4f} ({bal_acc*100:.2f}%)")
    print(f"  Precision:          {prec:.4f} ({prec*100:.2f}%)")
    print(f"  Sensitivity/Recall: {rec:.4f} ({rec*100:.2f}%)")
    print(f"  F1-Score:           {f1:.4f} ({f1*100:.2f}%)")
    print(f"  ROC-AUC:            {auc:.4f} ({auc*100:.2f}%)")
    print(f"  Specificity:        {spec:.4f} ({spec*100:.2f}%)")
    
    return {'name': name, 'accuracy': acc, 'balanced_accuracy': bal_acc, 
            'precision': prec, 'recall': rec, 'f1': f1, 'roc_auc': auc, 'specificity': spec}

# Get metrics for each model
results = []
results.append(get_model_metrics(lgbm, "LightGBM", X_test, y_test))
results.append(get_model_metrics(xgb, "XGBoost", X_test, y_test))
results.append(get_model_metrics(rf, "Random Forest", X_test, y_test))

 
print("\n" + "="*70)
print("TABLE 4: FINAL COMPARISON - ALL MODELS")
print("="*70)

print("\n{:<15} {:>10} {:>12} {:>10} {:>12} {:>10} {:>10}".format(
    "Model", "Accuracy", "Bal Acc", "Precision", "Recall", "F1-Score", "ROC-AUC"))
print("-" * 85)

for r in results:
    print("{:<15} {:>9.2f}% {:>11.2f}% {:>9.2f}% {:>11.2f}% {:>9.2f}% {:>9.2f}%".format(
        r['name'], r['accuracy']*100, r['balanced_accuracy']*100,
        r['precision']*100, r['recall']*100, r['f1']*100, r['roc_auc']*100))

print("-" * 85)
print("{:<15} {:>9.2f}% {:>11.2f}% {:>9.2f}% {:>11.2f}% {:>9.2f}% {:>9.2f}%".format(
    "Stacking (Opt)", acc_opt*100, bal_acc_opt*100, prec_opt*100, rec_opt*100, f1_opt*100, auc_stack*100))

# -------------------------------
# 21. TABLE 5: CONFUSION MATRICES
# -------------------------------
print("\n" + "="*70)
print("TABLE 5: CONFUSION MATRICES")
print("="*70)

print("\nConfusion Matrix - Stacking Ensemble (Default Threshold 0.5):")
print(f"                 Predicted: No    Predicted: Yes")
print(f"Actual: No       {cm_stack[0,0]:>10,}    {cm_stack[0,1]:>10,}")
print(f"Actual: Yes      {cm_stack[1,0]:>10,}    {cm_stack[1,1]:>10,}")

print("\nConfusion Matrix - Stacking Ensemble (Optimized Threshold {}):".format(best_thresh_acc))
print(f"                 Predicted: No    Predicted: Yes")
print(f"Actual: No       {cm_opt[0,0]:>10,}    {cm_opt[0,1]:>10,}")
print(f"Actual: Yes      {cm_opt[1,0]:>10,}    {cm_opt[1,1]:>10,}")

# -------------------------------
# 22. FINAL SUMMARY
# -------------------------------
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"\nBest Threshold: {best_thresh_acc:.2f}")
print(f"Final Optimized Accuracy: {best_acc*100:.2f}%")
print(f"Stacking Ensemble outperforms LightGBM by: {(acc_opt - results[0]['accuracy'])*100:.2f}%")
print(f"Stacking Ensemble outperforms XGBoost by: {(acc_opt - results[1]['accuracy'])*100:.2f}%")
print(f"Stacking Ensemble outperforms Random Forest by: {(acc_opt - results[2]['accuracy'])*100:.2f}%")

# -------------------------------
# 23. SAVE MODEL
# -------------------------------
joblib.dump(stacking_model, 'heart_disease_stacking_model.pkl')
print("\n✓ Model saved as 'heart_disease_stacking_model.pkl'")
# Add this at the end of your training code (after the model is trained)


# Save the preprocessor
joblib.dump(preprocessor, 'preprocessor.pkl')
print("✓ Preprocessor saved as 'preprocessor.pkl'")

# Save the optimal threshold
best_thresh_acc = 0.63 
joblib.dump(best_thresh_acc, 'threshold.pkl')
print("✓ Threshold saved as 'threshold.pkl'")

Dataset loaded - Shape: (319795, 18)

Features:
  Numerical: 21
  Categorical: 3
  Binary: 7
  Total Features: 31

Target distribution:
Class 0 (No): 292,422
Class 1 (Yes): 27,373
Imbalance ratio: 0.094

Processed features shape: (319795, 38)

After SMOTETomek:
Class 0: 288,157
Class 1: 171,188

Train set size: 367,476
Test set size: 91,869

TRAINING INDIVIDUAL MODELS...

Training LightGBM...
Training XGBoost...
Training Random Forest...

Building stacking ensemble...
Training stacking ensemble (this may take a few minutes)...

TABLE 1: STACKING ENSEMBLE RESULTS (DEFAULT THRESHOLD = 0.5)
  Accuracy:           0.9190 (91.90%)
  Balanced Accuracy:  0.9103 (91.03%)
  F1-Score:           0.8896 (88.96%)
  ROC-AUC:            0.9703 (97.03%)
  Precision:          0.9039 (90.39%)
  Sensitivity/Recall: 0.8758 (87.58%)
  Specificity:        0.9447 (94.47%)

THRESHOLD TUNING FOR MAX ACCURACY
Threshold: 0.45 | Accuracy: 0.9165 | F1: 0.8875
Threshold: 0.47 | Accuracy: 0.9175 | F1: 0.8884
Threshol